# **ML Model:** XGBoost

## **Notes:**

**XGBoost** (*eXtreme Gradient Boosting*) is boosting algorithm. Unlike **RF** (*Random Forest*) which builds trees parallel, boosting builds them sequentially and each of them tries to correct bad decisions of the previous one.

## **Implementation:**

#### Librabry imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import category_encoders as ce
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

### **Load data**

In [ ]:
data = pd.read_excel('../data/processed/accident_processed_srb.xlsx')
data.head(3)

,municipality,longitude,latitude,accident_type,description,month,day_of_week,hour,day_type,is_rush,is_night,season,acc_parked_vehicles,acc_pedestrians,acc_single_vehicle,acc_two_vehicles_no_turn,acc_two_vehicles_turn_or_cross
0,BARAJEVO,20.301589,44.568563,0,Nezgoda sa jednim vozilom – silazak sa kolovoz...,1,1,10,1,0,0,0,0,0,1,0,0
1,BARAJEVO,20.413280,44.579780,0,Najmanje dva vozila koja se kreću u istom smer...,1,3,12,1,0,0,0,0,0,0,1,0
2,BARAJEVO,20.312560,44.575470,0,Najmanje dva vozila koja se kreću istim putem ...,1,4,10,1,0,0,0,0,0,0,0,1


In [3]:
X = data.drop(columns='accident_type')
y = data['accident_type']

In [4]:
X.head(3)

,municipality,longitude,latitude,description,month,day_of_week,hour,day_type,is_rush,is_night,season,acc_parked_vehicles,acc_pedestrians,acc_single_vehicle,acc_two_vehicles_no_turn,acc_two_vehicles_turn_or_cross
0,BARAJEVO,20.301589,44.568563,Nezgoda sa jednim vozilom – silazak sa kolovoz...,1,1,10,1,0,0,0,0,0,1,0,0
1,BARAJEVO,20.413280,44.579780,Najmanje dva vozila koja se kreću u istom smer...,1,3,12,1,0,0,0,0,0,0,1,0
2,BARAJEVO,20.312560,44.575470,Najmanje dva vozila koja se kreću istim putem ...,1,4,10,1,0,0,0,0,0,0,0,1


In [5]:
y.head(3)

0    0
1    0
2    0
Name: accident_type, dtype: int64

### **Split data**

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [7]:
# Encoding after splitting data with fit just on train
mun_encoder = ce.TargetEncoder(cols=['municipality'])
X_train['municipality_encoded'] = mun_encoder.fit_transform(X_train['municipality'], y_train)
X_test['municipality_encoded'] = mun_encoder.transform(X_test['municipality'])

desc_encoder = ce.TargetEncoder(cols=['description'])
X_train['description_encoded'] = desc_encoder.fit_transform(X_train['description'], y_train)
X_test['description_encoded'] = desc_encoder.transform(X_test['description'])

X_train = X_train.drop(columns=['municipality', 'description'])
X_test = X_test.drop(columns=['municipality', 'description'])

### **Training**

In [8]:
import xgboost as xgb

In [9]:
xgb_model = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    reg_alpha=0.0,
    objective='multi:softmax',
    num_class=len(y.unique()),
    eval_metric='mlogloss',
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1
)

In [10]:
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=50
)

print(f"\nOptimalan broj stabala: {xgb_model.best_iteration}")

[0]	validation_0-mlogloss:0.66162	validation_1-mlogloss:0.66160
[50]	validation_0-mlogloss:0.48463	validation_1-mlogloss:0.48644
[100]	validation_0-mlogloss:0.47419	validation_1-mlogloss:0.47861
[150]	validation_0-mlogloss:0.46933	validation_1-mlogloss:0.47679
[200]	validation_0-mlogloss:0.46537	validation_1-mlogloss:0.47578
[250]	validation_0-mlogloss:0.46207	validation_1-mlogloss:0.47520
[300]	validation_0-mlogloss:0.45907	validation_1-mlogloss:0.47484
[350]	validation_0-mlogloss:0.45609	validation_1-mlogloss:0.47470
[400]	validation_0-mlogloss:0.45338	validation_1-mlogloss:0.47452
[450]	validation_0-mlogloss:0.45059	validation_1-mlogloss:0.47443
[500]	validation_0-mlogloss:0.44802	validation_1-mlogloss:0.47438
[550]	validation_0-mlogloss:0.44560	validation_1-mlogloss:0.47433
[570]	validation_0-mlogloss:0.44468	validation_1-mlogloss:0.47441

Optimalan broj stabala: 520


### **Evaluation**

In [11]:
y_pred = xgb_model.predict(X_test)

In [12]:
print("Classification Report:")
print(classification_report(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.83      0.80     24509
           1       0.72      0.65      0.68     16562

    accuracy                           0.76     41071
   macro avg       0.75      0.74      0.74     41071
weighted avg       0.76      0.76      0.76     41071



In [13]:
f1 = f1_score(y_test, y_pred, average='weighted')
print(f"Weighted F1 Score: {f1:.4f}")

Weighted F1 Score: 0.7552


### **Additional:**

**Note:**
- Last score: `Weighted F1 Score: 0.7553`
- Last repost: 
  ```
  Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.83      0.80     24509
           1       0.72      0.65      0.68     16562

    accuracy                           0.76     41071
   macro avg       0.75      0.74      0.74     41071
  weighted avg       0.76      0.76      0.76     41071
  ```